# Piece markers


## Import


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from loguru import logger as lg
from rich import get_console
from rich import print as rprint
from rich.console import Console

console: Console = get_console()
console.is_jupyter = False

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from snap_fit.aruco.sheet_metadata import QRChunkHandler
from snap_fit.aruco.sheet_metadata import SheetMetadata
from snap_fit.aruco.slot_grid import SlotGrid
from snap_fit.config.aruco.aruco_board_config import ArucoBoardConfig
from snap_fit.config.aruco.metadata_zone_config import MetadataZoneConfig
from snap_fit.config.aruco.metadata_zone_config import SlotGridConfig
from snap_fit.config.aruco.sheet_aruco_config import SheetArucoConfig
from snap_fit.params.snap_fit_params import get_snap_fit_params

## Params and config


In [ ]:
project_name_params = get_snap_fit_params()
rprint(project_name_params)

## Showcase


### Step 02: SheetMetadata model


In [ ]:
meta = SheetMetadata(
    tag_name="oca",
    sheet_index=1,
    total_sheets=6,
    board_config_id="oca",
)
rprint(meta)

In [ ]:
payload = meta.to_qr_payload()
print(f"Payload: {payload!r}  ({len(payload)} bytes)")

recovered = SheetMetadata.from_qr_payload(payload)
assert recovered == meta
print("Round-trip OK")

In [ ]:
meta_unknown = SheetMetadata(
    tag_name="milano1",
    sheet_index=0,
    total_sheets=None,
    board_config_id="ab1",
)
payload_unknown = meta_unknown.to_qr_payload()
print(f"Unknown total_sheets payload: {payload_unknown!r}")
assert SheetMetadata.from_qr_payload(payload_unknown) == meta_unknown
print("Round-trip OK")

### Step 03: QRChunkHandler encode / decode


In [ ]:
handler = QRChunkHandler(n_codes=3, ecc="M")
images = handler.encode(payload)
print(f"Generated {len(images)} QR images, each {images[0].shape}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, img in zip(axes, images):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.axis("off")
fig.suptitle("3x identical QR codes (full payload each)")
plt.tight_layout()
plt.show()

In [ ]:
for i, img in enumerate(images):
    decoded = handler.decode_first(img)
    assert decoded == payload, f"Code {i} failed: {decoded!r}"
    print(f"Code {i}: decoded OK -> {decoded!r}")

In [ ]:
blank = np.ones((100, 100), dtype=np.uint8) * 255
result = handler.decode_first(blank)
print(f"Blank image decode: {result!r}")

### Step 04: MetadataZoneConfig + SlotGridConfig


In [ ]:
zone = MetadataZoneConfig()
rprint(zone)

In [ ]:
custom_zone = MetadataZoneConfig(
    qr_n_codes=3,
    qr_ecc="M",
    text_enabled=True,
    slot_grid=SlotGridConfig(cols=10, rows=8, label_inset_px=6),
)
rprint(custom_zone)

In [ ]:
json_str = zone.model_dump_json()
rprint(json_str)
recovered_zone = MetadataZoneConfig.model_validate_json(json_str)
assert recovered_zone == zone
rprint("JSON round-trip OK")

In [ ]:
cfg_default = SheetArucoConfig.model_validate({"detector": {}})
print(f"metadata_zone default: {cfg_default.metadata_zone!r}")

cfg_with_zone = SheetArucoConfig.model_validate(
    {
        "detector": {},
        "metadata_zone": {
            "qr_n_codes": 3,
            "slot_grid": {"cols": 8, "rows": 6},
        },
    }
)
rprint(cfg_with_zone)

In [ ]:
import json
from pathlib import Path

params = get_snap_fit_params()
oca_cfg_path = params.paths.data_fol / "oca" / "oca_SheetArucoConfig.json"
if oca_cfg_path.exists():
    raw = json.loads(oca_cfg_path.read_text())
    loaded = SheetArucoConfig.model_validate(raw)
    print(f"oca config loaded OK, metadata_zone={loaded.metadata_zone!r}")
else:
    print(f"File not found (expected in dev): {oca_cfg_path}")

### Step 05: SlotGrid geometry + label rendering


In [ ]:
# SlotGrid with default board and 8x6 grid
board_cfg = ArucoBoardConfig()
grid_cfg = SlotGridConfig(cols=8, rows=6)
sg = SlotGrid(grid_cfg, board_cfg)

board_w, board_h = board_cfg.board_dimensions()
print(f"Board dimensions: {board_w} x {board_h}")
print(
    f"Interior x: {sg._interior_x0} .. {sg._interior_x1}  ({sg._interior_x1 - sg._interior_x0} px)"
)
print(
    f"Interior y: {sg._interior_y0} .. {sg._interior_y1}  ({sg._interior_y1 - sg._interior_y0} px)"
)
print(
    f"Piece area y1: {sg._piece_area_y1}  (QR strip y={sg._piece_area_y1}..{sg._interior_y1})"
)
print(f"Slot size: {sg._slot_w:.1f} x {sg._slot_h:.1f} px")

In [ ]:
# Label grid: first, last, middle slots
print("label_for_slot(0, 0):", sg.label_for_slot(0, 0))  # A1
print("label_for_slot(7, 5):", sg.label_for_slot(7, 5))  # H6
print("label_for_slot(2, 3):", sg.label_for_slot(2, 3))  # C4

# Full label grid
print("\nFull 8x6 label grid:")
for row in range(grid_cfg.rows):
    row_labels = [sg.label_for_slot(col, row) for col in range(grid_cfg.cols)]
    print("  ", " ".join(f"{lbl:3}" for lbl in row_labels))

In [ ]:
# Verify slot_centers round-trip via slot_for_centroid
centers = sg.slot_centers()
assert len(centers) == grid_cfg.cols * grid_cfg.rows
for idx, (cx, cy) in enumerate(centers):
    expected = (idx % grid_cfg.cols, idx // grid_cfg.cols)
    result = sg.slot_for_centroid(cx, cy)
    assert result == expected, f"Center {idx} failed: {result} != {expected}"
print(f"All {len(centers)} slot centres map back to their own slot OK")

# Out-of-bounds checks
assert sg.slot_for_centroid(0, 500) is None  # far left (in ring)
assert sg.slot_for_centroid(500, sg._piece_area_y1 + 10) is None  # in QR strip
print("Out-of-bounds checks OK")

In [ ]:
# Visual: render_labels on a white board-sized image
from snap_fit.aruco.aruco_board import ArucoBoardGenerator

board_gen = ArucoBoardGenerator(board_cfg)
board_img_gray = board_gen.generate_image()

# Convert to BGR for coloured text
board_img_bgr = np.stack([board_img_gray] * 3, axis=-1)
labelled = sg.render_labels(board_img_bgr)

# Draw piece area boundary for reference (green rectangle)
import cv2

labelled_vis = labelled.copy()
cv2.rectangle(
    labelled_vis,
    (sg._interior_x0, sg._interior_y0),
    (sg._interior_x1, sg._piece_area_y1),
    (0, 180, 0),
    2,
)
# Draw QR strip boundary (blue)
cv2.rectangle(
    labelled_vis,
    (sg._interior_x0, sg._piece_area_y1),
    (sg._interior_x1, sg._interior_y1),
    (200, 100, 0),
    2,
)

plt.figure(figsize=(6, 9))
plt.imshow(cv2.cvtColor(labelled_vis, cv2.COLOR_BGR2RGB))
plt.title("Board with slot labels (green=piece area, blue=QR strip)")
plt.axis("off")
plt.tight_layout()
plt.show()

### Step 06: SheetMetadataEncoder + SheetMetadataDecoder


In [ ]:
from snap_fit.aruco.sheet_metadata import SheetMetadataDecoder
from snap_fit.aruco.sheet_metadata import SheetMetadataEncoder

board_cfg = ArucoBoardConfig()
zone_cfg = MetadataZoneConfig()
encoder = SheetMetadataEncoder(board_cfg)

strip = encoder._strip_region()
x0, y0, x1, y1 = strip
print(f"QR strip region: x={x0}..{x1}, y={y0}..{y1}  ({x1 - x0} x {y1 - y0} px)")

In [ ]:
# Render metadata onto a white board-sized image
meta = SheetMetadata(
    tag_name="oca",
    sheet_index=1,
    total_sheets=6,
    board_config_id="oca",
)

board_w, board_h = board_cfg.board_dimensions()
blank = np.full((board_h, board_w, 3), 255, dtype=np.uint8)
encoded = encoder.render(blank, meta, zone_cfg)

# Visualise: zoom into the QR strip area with some margin
margin = 20
y_vis0 = max(0, y0 - margin - 30)  # include text line above strip
y_vis1 = min(board_h, y1 + margin)
strip_vis = encoded[y_vis0:y_vis1, x0:x1]

plt.figure(figsize=(10, 3))
plt.imshow(cv2.cvtColor(strip_vis, cv2.COLOR_BGR2RGB))
plt.title("QR strip + text line (zoomed)")
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Encode-decode round-trip
decoder = SheetMetadataDecoder()
recovered = decoder.decode(encoded)
print(f"Decoded metadata: {recovered}")
assert recovered == meta, f"Round-trip failed: {recovered} != {meta}"
print("Round-trip OK")

In [ ]:
# Decoder on blank image - should return None without raising
blank_small = np.full((200, 200, 3), 255, dtype=np.uint8)
result = decoder.decode(blank_small)
print(f"No-QR decode result: {result!r}")  # expected: None

### Step 07 preliminary: dimension computation


In [ ]:
# === Dimension debug: targeted x-coordinate analysis ===
import numpy as np
from io import StringIO
from pathlib import Path
from snap_fit.aruco.aruco_board import ArucoBoardGenerator
from snap_fit.aruco.sheet_metadata import SheetMetadataEncoder
from snap_fit.aruco.slot_grid import SlotGrid
from snap_fit.config.aruco.aruco_board_config import ArucoBoardConfig
from snap_fit.config.aruco.metadata_zone_config import MetadataZoneConfig

_buf = StringIO()


def _p(*args, **kw):
    print(*args, **kw, file=_buf)


cfg = ArucoBoardConfig()
zone = MetadataZoneConfig()
grid_cfg = zone.slot_grid
board_w, board_h = cfg.board_dimensions()
stride = cfg.marker_length + cfg.marker_separation  # 200

sg = SlotGrid(grid_cfg, cfg)
encoder = SheetMetadataEncoder(cfg)
strip = encoder._strip_region()
qr_x0, qr_y0, qr_x1, qr_y1 = strip

gen = ArucoBoardGenerator(cfg)
aruco_gray = gen.generate_image()
aruco_mask = aruco_gray < 200

_p("=== BOARD DIMENSIONS ===")
_p(f"board = {board_w} x {board_h}")
_p(
    f"margin={cfg.margin}, marker_length={cfg.marker_length}, marker_sep={cfg.marker_separation}"
)
_p(f"stride={stride}, border_bits={cfg.border_bits}")
_p()

_p("=== INTERIOR / PIECE AREA / QR STRIP ===")
_p(
    f"interior  = x[{sg._interior_x0}, {sg._interior_x1}] y[{sg._interior_y0}, {sg._interior_y1}]  {sg._interior_x1 - sg._interior_x0}x{sg._interior_y1 - sg._interior_y0}"
)
_p(
    f"piece_area= x[{sg._interior_x0}, {sg._interior_x1}] y[{sg._interior_y0}, {sg._piece_area_y1}]  {sg._interior_x1 - sg._interior_x0}x{sg._piece_area_y1 - sg._interior_y0}"
)
_p(
    f"qr_strip  = x[{qr_x0}, {qr_x1}] y[{qr_y0}, {qr_y1}]  {qr_x1 - qr_x0}x{qr_y1 - qr_y0}"
)
_p(f"slot_w={sg._slot_w:.2f}, slot_h={sg._slot_h:.2f}")
ring_start = cfg.margin + cfg.marker_length
_p(f"ring_start={ring_start}")
_p()

_p("=== RING BAND ASYMMETRY ===")
_p(
    f"left_band  = {sg._interior_x0} px (margin+marker = {cfg.margin}+{cfg.marker_length})"
)
_p(f"right_band = {board_w - sg._interior_x1} px (marker only = {cfg.marker_length})")
_p(
    f"top_band   = {sg._interior_y0} px (margin+marker = {cfg.margin}+{cfg.marker_length})"
)
_p(f"bottom_band= {board_h - sg._interior_y1} px (marker only = {cfg.marker_length})")
_p(f"  NOTE: 20 px ASYMMETRY between left/right and top/bottom!")
_p()

# === PIXEL-LEVEL: Find actual marker bounding boxes ===
_p("=== ACTUAL MARKER PIXEL BOUNDING BOXES (from rendered image) ===")
# Find the actual min/max x for all marker rows within the grid
# Check row 1 side markers: expected y=[220, 320] at x=[20, 120] and x=[820, 920]
sample_rows = [
    (1, "row 1 side markers"),
    (3, "row 3 side markers"),
    (5, "row 5 side markers (near QR strip)"),
    (6, "row 6 bottom markers"),
]
for grid_row, desc in sample_rows:
    expected_y0 = cfg.margin + grid_row * stride
    expected_y1 = expected_y0 + cfg.marker_length
    _p(f"\n  {desc}: expected y=[{expected_y0}, {expected_y1}]")

    # Scan a window around the expected y range
    scan_y0 = max(0, expected_y0 - 25)
    scan_y1 = min(board_h, expected_y1 + 25)

    # Find the actual y range with marker pixels
    actual_y_min = None
    actual_y_max = None
    for y in range(scan_y0, scan_y1):
        if aruco_mask[y, :].sum() > 0:
            if actual_y_min is None:
                actual_y_min = y
            actual_y_max = y
    if actual_y_min is not None:
        _p(
            f"    actual y range: [{actual_y_min}, {actual_y_max + 1}] (expected [{expected_y0}, {expected_y1}])"
        )
        _p(
            f"    y overshoot top: {expected_y0 - actual_y_min} px, bottom: {actual_y_max + 1 - expected_y1} px"
        )

    # Now check x-coordinates of marker pixels at the middle of this row
    mid_y = (expected_y0 + expected_y1) // 2
    x_pixels = np.where(aruco_mask[mid_y, :])[0]
    if len(x_pixels) > 0:
        # Find contiguous runs
        runs = np.split(x_pixels, np.where(np.diff(x_pixels) > 5)[0] + 1)
        _p(
            f"    marker x-runs at y={mid_y}: {[(r[0], r[-1] + 1) for r in runs if len(r) > 0]}"
        )
        for run in runs:
            if len(run) > 0:
                rx0, rx1 = run[0], run[-1] + 1
                in_interior = rx0 < sg._interior_x1 and rx1 > sg._interior_x0
                overlap_x0 = max(rx0, sg._interior_x0)
                overlap_x1 = min(rx1, sg._interior_x1)
                if overlap_x1 > overlap_x0:
                    _p(
                        f"      run [{rx0}, {rx1}] OVERLAPS interior [{sg._interior_x0}, {sg._interior_x1}] by {overlap_x1 - overlap_x0} px"
                    )

    # Check specifically inside the interior at this row
    interior_marker_count = aruco_mask[mid_y, sg._interior_x0 : sg._interior_x1].sum()
    if interior_marker_count > 0:
        interior_x_pixels = (
            np.where(aruco_mask[mid_y, sg._interior_x0 : sg._interior_x1])[0]
            + sg._interior_x0
        )
        _p(f"    interior marker pixels at y={mid_y}: x={interior_x_pixels.tolist()}")

_p()
_p("=== DETAILED OVERLAP ANALYSIS ===")

# Check actual marker extent at the right boundary
_p("\n  RIGHT BOUNDARY analysis (near x=820):")
for y_check in range(200, 1220, 200):  # check at each marker row
    row_pixels_near_right = np.where(aruco_mask[y_check, 800:840])[0] + 800
    if len(row_pixels_near_right) > 0:
        _p(
            f"    y={y_check}: marker px near right boundary: x={row_pixels_near_right.tolist()}"
        )

_p("\n  BOTTOM BOUNDARY analysis (near y=1220):")
for x_check in [220, 420, 620]:  # check at bottom marker columns
    col_pixels_near_bottom = np.where(aruco_mask[1190:1250, x_check])[0] + 1190
    if len(col_pixels_near_bottom) > 0:
        _p(
            f"    x={x_check}: marker px near bottom: y={col_pixels_near_bottom.tolist()}"
        )

# QR strip specific overlap
_p("\n  QR STRIP MARKER OVERLAP:")
qr_marker_count = aruco_mask[qr_y0:qr_y1, qr_x0:qr_x1].sum()
_p(f"  Total marker pixels in QR strip region: {qr_marker_count}")
if qr_marker_count > 0:
    for y in range(qr_y0, qr_y1):
        c = aruco_mask[y, qr_x0:qr_x1].sum()
        if c > 0:
            x_pos = np.where(aruco_mask[y, qr_x0:qr_x1])[0] + qr_x0
            runs = np.split(x_pos, np.where(np.diff(x_pos) > 5)[0] + 1)
            run_summary = [(r[0], r[-1] + 1) for r in runs if len(r) > 0]
            _p(f"    y={y}: {c} px, runs={run_summary}")

# Summary statistics
_p("\n=== OVERLAP SUMMARY ===")
total_interior = aruco_mask[
    sg._interior_y0 : sg._interior_y1, sg._interior_x0 : sg._interior_x1
].sum()
total_piece = aruco_mask[
    sg._interior_y0 : sg._piece_area_y1, sg._interior_x0 : sg._interior_x1
].sum()
total_qr = aruco_mask[qr_y0:qr_y1, qr_x0:qr_x1].sum()
_p(f"  Marker pixels in interior:   {total_interior}")
_p(f"  Marker pixels in piece area: {total_piece}")
_p(f"  Marker pixels in QR strip:   {total_qr}")

_p()
_p("=== TEXT LINE POSITION ===")
text_y = qr_y0 - 4
_p(f"  Text baseline y={text_y}, piece area bottom y={sg._piece_area_y1}")
_p(f"  Text approx top y={text_y - 12}")
_p(
    f"  Text is {sg._piece_area_y1 - (text_y - 12)} px ABOVE piece area bottom (positive=safe, negative=overlap)"
)
actual_gap = sg._piece_area_y1 - text_y
_p(f"  Baseline to piece bottom gap: {actual_gap} px")

_p()
_p("=== REGION SUMMARY TABLE ===")
_p(f"  {'Region':<20} {'x0':>5} {'y0':>5} {'x1':>5} {'y1':>5} {'W':>5} {'H':>5}")
_p("  " + "-" * 60)
for name, x0_, y0_, x1_, y1_ in [
    ("Board", 0, 0, board_w, board_h),
    ("Left ring band", 0, 0, sg._interior_x0, board_h),
    ("Right ring band", sg._interior_x1, 0, board_w, board_h),
    ("Top ring band", 0, 0, board_w, sg._interior_y0),
    ("Bottom ring band", 0, sg._interior_y1, board_w, board_h),
    ("Interior", sg._interior_x0, sg._interior_y0, sg._interior_x1, sg._interior_y1),
    (
        "Piece area",
        sg._interior_x0,
        sg._interior_y0,
        sg._interior_x1,
        sg._piece_area_y1,
    ),
    ("QR strip", qr_x0, qr_y0, qr_x1, qr_y1),
]:
    _p(
        f"  {name:<20} {x0_:>5} {y0_:>5} {x1_:>5} {y1_:>5} {x1_ - x0_:>5} {y1_ - y0_:>5}"
    )

out_path = Path("12_dim_debug_output.txt")
out_path.write_text(_buf.getvalue())
print(f"Output written to {out_path}")
print(_buf.getvalue())

### Step 07: BoardImageComposer full assembly


In [ ]:
from snap_fit.aruco.board_image_composer import BoardImageComposer

In [ ]:
# Base compose: no metadata_zone -> plain ArUco board (BGR)
board_cfg = ArucoBoardConfig()
composer_plain = BoardImageComposer(board_cfg)
plain_img = composer_plain.compose()
print(f"Plain board shape: {plain_img.shape}  dtype={plain_img.dtype}")
assert plain_img.ndim == 3, "Expected BGR image"

plt.figure(figsize=(5, 7))
plt.imshow(cv2.cvtColor(plain_img, cv2.COLOR_BGR2RGB))
plt.title("Plain ArUco board (no metadata zone)")
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Compose with slot grid only (no metadata -> QR strip skipped)
zone_cfg = MetadataZoneConfig()
composer_grid = BoardImageComposer(board_cfg, zone_cfg)
grid_img = composer_grid.compose()

plt.figure(figsize=(5, 7))
plt.imshow(cv2.cvtColor(grid_img, cv2.COLOR_BGR2RGB))
plt.title("Board with slot grid labels (no QR)")
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Full compose: ArUco ring + slot labels + QR strip + text
from datetime import date

meta_full = SheetMetadata(
    tag_name="oca",
    sheet_index=1,
    total_sheets=6,
    board_config_id="oca",
    printed_at=date(2025, 1, 15),
)

full_img = composer_grid.compose(meta_full)

plt.figure(figsize=(5, 7))
plt.imshow(cv2.cvtColor(full_img, cv2.COLOR_BGR2RGB))
plt.title("Full board: ArUco + slot labels + QR + text")
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Decode round-trip: compose -> SheetMetadataDecoder -> assert
from snap_fit.aruco.sheet_metadata import SheetMetadataDecoder

decoder = SheetMetadataDecoder()
recovered = decoder.decode(full_img)
assert recovered == meta_full, f"Round-trip failed: {recovered}"
print(f"Round-trip OK: {recovered}")

# Verify output dimensions match board config
board_w, board_h = board_cfg.board_dimensions()
assert full_img.shape == (board_h, board_w, 3), f"Unexpected shape: {full_img.shape}"
print(f"Dimensions OK: {full_img.shape}")

### Step 08: Contour.centroid property


### Step 09: Sheet integration - metadata + slot labels in pipeline


In [ ]:
# 9a: Sheet with metadata and slot_grid attached directly (no SheetAruco)
import tempfile

import cv2

from snap_fit.aruco.board_image_composer import BoardImageComposer
from snap_fit.aruco.sheet_metadata import SheetMetadata
from snap_fit.aruco.slot_grid import SlotGrid
from snap_fit.config.aruco.aruco_board_config import ArucoBoardConfig
from snap_fit.config.aruco.metadata_zone_config import MetadataZoneConfig
from snap_fit.config.aruco.metadata_zone_config import SlotGridConfig
from snap_fit.puzzle.sheet import Sheet

board_cfg = ArucoBoardConfig()
zone_cfg = MetadataZoneConfig()
grid_cfg = SlotGridConfig(cols=4, rows=3)

meta = SheetMetadata(
    tag_name="oca", sheet_index=1, total_sheets=6, board_config_id="oca"
)

# Compose a full board image with QR + grid
composer = BoardImageComposer(board_cfg, MetadataZoneConfig(slot_grid=grid_cfg))
board_img = composer.compose(meta)

# Save to a temp path so Sheet can load it
with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as f:
    tmp_path = Path(f.name)

cv2.imwrite(str(tmp_path), board_img)
print(f"Saved composed board to {tmp_path}")


In [ ]:
# 9b: Sheet attributes - metadata None, slot_grid None by default
plain_sheet = Sheet(img_fp=tmp_path, min_area=0, image=board_img)
print(f"metadata: {plain_sheet.metadata!r}")
print(f"slot_grid: {plain_sheet.slot_grid!r}")
print(
    f"First piece label: {plain_sheet.pieces[0].label if plain_sheet.pieces else '(no pieces)'}"
)


In [ ]:
# 9c: Sheet with slot_grid passed in - pieces get slot labels
slot_grid = SlotGrid(grid_cfg, board_cfg)
labelled_sheet = Sheet(
    img_fp=tmp_path, min_area=0, image=board_img, slot_grid=slot_grid
)
print(f"slot_grid set: {labelled_sheet.slot_grid is not None}")
for p in labelled_sheet.pieces[:6]:
    cx, cy = p.contour.centroid
    print(f"  piece {p.piece_id.piece_id}  centroid=({cx},{cy})  label={p.label!r}")


In [ ]:
# 9d: SheetRecord round-trip with metadata
from snap_fit.data_models.sheet_record import SheetRecord

sheet_for_record = Sheet(img_fp=tmp_path, min_area=0, image=board_img)
sheet_for_record.metadata = meta

record = SheetRecord.from_sheet(sheet_for_record)
print(f"SheetRecord.metadata: {record.metadata!r}")

json_str = record.model_dump_json()
recovered = SheetRecord.model_validate_json(json_str)
assert recovered.metadata == meta
print("SheetRecord JSON round-trip with metadata OK")


In [ ]:
# 9e: PieceRecord round-trip with label
from snap_fit.data_models.piece_record import PieceRecord

if labelled_sheet.pieces:
    piece_with_label = labelled_sheet.pieces[0]
    piece_with_label.label = "A1"  # manually assign for demo
    prec = PieceRecord.from_piece(piece_with_label)
    print(f"PieceRecord.label: {prec.label!r}")
    json_str = prec.model_dump_json()
    recovered_prec = PieceRecord.model_validate_json(json_str)
    assert recovered_prec.label == "A1"
    print("PieceRecord JSON round-trip with label OK")
else:
    print("No pieces detected (composed board has no puzzle pieces)")


In [ ]:
# 9f: SheetAruco end-to-end with metadata_zone config
from snap_fit.config.aruco.aruco_detector_config import ArucoDetectorConfig
from snap_fit.config.aruco.sheet_aruco_config import SheetArucoConfig
from snap_fit.puzzle.sheet_aruco import SheetAruco

cfg = SheetArucoConfig(
    min_area=0,
    detector=ArucoDetectorConfig(),
    metadata_zone=MetadataZoneConfig(slot_grid=grid_cfg),
)
aruco_loader = SheetAruco(cfg)

# load_sheet on the composed board (no ArUco markers - rectification will fail
# and fall back to original, but metadata decode happens pre-rectification)
loaded_sheet = aruco_loader.load_sheet(tmp_path)
print(f"sheet.metadata: {loaded_sheet.metadata!r}")
print(f"sheet.slot_grid is not None: {loaded_sheet.slot_grid is not None}")

# Cleanup temp file
tmp_path.unlink(missing_ok=True)


In [ ]:
import numpy as np

from snap_fit.image.contour import Contour

# Square contour: top-left (100, 200), size 80 - centroid expected at (140, 240)
pts_square = np.array(
    [[[100, 200]], [[180, 200]], [[180, 280]], [[100, 280]]], dtype=np.int32
)
square = Contour(pts_square)
cx, cy = square.centroid
print(f"Square centroid: ({cx}, {cy})  (expected ~140, 240)")

# Circle contour at (300, 400) r=60
angles = np.linspace(0, 2 * np.pi, 64, endpoint=False)
pts_circle = np.round(
    np.stack([300 + 60 * np.cos(angles), 400 + 60 * np.sin(angles)], axis=1)
).astype(np.int32)[:, np.newaxis, :]
circle = Contour(pts_circle)
cx2, cy2 = circle.centroid
print(f"Circle centroid: ({cx2}, {cy2})  (expected ~300, 400)")

# Degenerate single-point contour - should not raise
pts_degen = np.array([[[50, 60]]], dtype=np.int32)
degen = Contour(pts_degen)
cx3, cy3 = degen.centroid
print(f"Degenerate centroid (fallback): ({cx3}, {cy3})")